# Tessera IR Pipeline Tutorial

This notebook executes the canonical `@tessera.jit` CPU path and inspects the real Graph, Schedule, Tile, and Target IR artifacts. It intentionally fails instead of substituting a shim or illustrative output.

In [ ]:
import numpy as np
import tessera as ts

mlp_step = ts.from_text("""
    def mlp_step(x, w):
        return ts.ops.relu(ts.ops.matmul(x, w))
""", cpu_tile=(32, 32, 16))

x = np.arange(16, dtype=np.float32).reshape(4, 4)
w = np.eye(4, dtype=np.float32)
out = mlp_step(x, w)
np.testing.assert_array_equal(out, x)
print(mlp_step.explain())

## Inspect the emitted compiler artifacts

Each value below comes from the compiled function. Empty stages are treated as a regression.

In [ ]:
artifacts = {
    'graph': mlp_step.ir_text(),
    'schedule': mlp_step.schedule_ir,
    'tile': mlp_step.tile_ir,
    'target': mlp_step.target_ir,
}
missing = [level for level, text in artifacts.items() if not text]
assert not missing, f'empty compiler artifacts: {missing}'
for level, text in artifacts.items():
    print(f'\n==== {level.upper()} IR ({len(text)} chars) ====')
    print(text[:1200])

## Query support without claiming device execution

The tutorial executed the CPU reference path. Target rows below are capability metadata, not exact-device proof.

In [ ]:
for op_name in ('matmul', 'relu'):
    info = ts.compiler.support(op_name)
    print(op_name, info.best_tier.value)
    for target in info.targets:
        print(' ', target.target, target.target_ir, target.runtime, target.tier.value)